# Trace-native memory as versioned database programs

The append-only ledger is the authority. Memory is a reproducible view over ledger evidence, not a second mutable story. This notebook uses standard-library SQLite only so it runs everywhere. The harness uses byte-equal JSONL views by default, optional DuckDB for analytical execution, and optional LanceDB as a disposable search projection.

In [ ]:
import hashlib
import json
import sqlite3


def canonical(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


db = sqlite3.connect(":memory:")
db.row_factory = sqlite3.Row
db.executescript("""
CREATE TABLE events (seq INTEGER PRIMARY KEY, observed_at TEXT NOT NULL, kind TEXT NOT NULL, status TEXT NOT NULL, subject TEXT, text TEXT NOT NULL, payload_json TEXT NOT NULL);
CREATE TABLE view_cache (cache_key TEXT PRIMARY KEY, program TEXT NOT NULL, version INTEGER NOT NULL, watermark INTEGER NOT NULL, value_json TEXT NOT NULL, content_hash TEXT NOT NULL);
""")
rows = [
    (
        1,
        "2026-09-01T20:30:00Z",
        "task.started",
        "completed",
        "task",
        "Fix parser timeout",
        {"goal": "Fix parser timeout"},
    ),
    (
        2,
        "2026-09-01T20:31:00Z",
        "fact.asserted",
        "completed",
        "repo.test_command",
        "pytest -q tests/test_parser.py",
        {"confidence": 1.0},
    ),
    (
        3,
        "2026-09-01T20:32:00Z",
        "file.read",
        "completed",
        "parser.py",
        "recursive union parser inspected",
        {},
    ),
    (
        4,
        "2026-09-01T20:34:00Z",
        "test.run",
        "timeout",
        "tests/test_parser.py",
        "parser test timed out after 30 seconds",
        {"effect": "unknown"},
    ),
    (
        5,
        "2026-09-01T20:35:00Z",
        "memory.semantic",
        "completed",
        "parser.recursion",
        "recursive parser branches need explicit depth or cycle guards",
        {"derived_from": [3, 4]},
    ),
]
db.executemany(
    "INSERT INTO events VALUES (?, ?, ?, ?, ?, ?, ?)",
    [(*row[:6], canonical(row[6])) for row in rows],
)
assert db.execute("SELECT count(*) FROM events").fetchone()[0] == 5

## Stored-program contract

A program is identified by name and version. A result is identified by the program, version, task watermark, and canonical arguments. It returns canonical bytes, a content hash, and evidence sequence IDs. Updating logic creates version 2; it never changes the meaning of a cached version-1 result.

In [ ]:
PROGRAMS = {
    (
        "memory.factual",
        1,
    ): """SELECT subject, text, max(seq) AS seq FROM events WHERE kind='fact.asserted' AND status='completed' GROUP BY subject ORDER BY subject""",
    (
        "memory.episodic",
        1,
    ): """SELECT seq, observed_at, kind, status, text FROM events ORDER BY seq DESC LIMIT :limit""",
    (
        "memory.semantic",
        1,
    ): """SELECT seq, subject, text FROM events WHERE kind='memory.semantic' AND status='completed' AND lower(text) LIKE :query ORDER BY seq DESC""",
    (
        "task.progress",
        1,
    ): """SELECT status, count(*) AS count FROM events GROUP BY status ORDER BY status""",
}


def run_program(name, *, version=1, **arguments):
    sql = PROGRAMS[(name, version)]
    watermark = db.execute("SELECT coalesce(max(seq), 0) FROM events").fetchone()[0]
    params = dict(arguments)
    if name == "memory.episodic":
        params.setdefault("limit", 20)
    if name == "memory.semantic":
        params["query"] = "%" + params.get("query", "").casefold() + "%"
    identity = {"program": name, "version": version, "watermark": watermark, "arguments": arguments}
    cache_key = hashlib.sha256(canonical(identity).encode()).hexdigest()
    cached = db.execute(
        "SELECT value_json, content_hash FROM view_cache WHERE cache_key=?", (cache_key,)
    ).fetchone()
    if cached:
        data = json.loads(cached["value_json"])
        evidence_seqs = sorted({row["seq"] for row in data if "seq" in row}) or list(
            range(1, watermark + 1)
        )
        return {
            "cache": "hit",
            "cache_key": cache_key,
            "content_hash": cached["content_hash"],
            "evidence_seqs": evidence_seqs,
            "data": data,
            **identity,
        }
    data = [dict(row) for row in db.execute(sql, params)]
    value_json = canonical(data)
    content_hash = hashlib.sha256(value_json.encode()).hexdigest()
    db.execute(
        "INSERT INTO view_cache VALUES (?, ?, ?, ?, ?, ?)",
        (cache_key, name, version, watermark, value_json, content_hash),
    )
    evidence_seqs = sorted({row["seq"] for row in data if "seq" in row}) or list(
        range(1, watermark + 1)
    )
    return {
        "cache": "miss",
        "cache_key": cache_key,
        "content_hash": content_hash,
        "evidence_seqs": evidence_seqs,
        "data": data,
        **identity,
    }


first = run_program("memory.semantic", query="recursive")
second = run_program("memory.semantic", query="recursive")
assert first["cache"] == "miss" and second["cache"] == "hit"
assert first["data"] == second["data"] and first["content_hash"] == second["content_hash"]
print(first["data"], {"first": first["cache"], "second": second["cache"]})

## Factual, episodic, and semantic memory

- **Factual:** current claims such as the verified test command. Favor latest supported truth and retain provenance.
- **Episodic:** ordered events around what happened, including failures and elapsed time.
- **Semantic:** generalized concepts derived from evidence, such as a parser-recursion lesson. The mock uses lexical SQL; production semantic/vector retrieval belongs in a disposable Lance projection that returns canonical event IDs.

These are projections over the same ledger, so deleting or correcting the authoritative evidence can deterministically rebuild every view.

In [ ]:
views = {
    "factual": run_program("memory.factual"),
    "episodic": run_program("memory.episodic", limit=3),
    "semantic": run_program("memory.semantic", query="cycle guards"),
    "progress": run_program("task.progress"),
}
for name, result in views.items():
    print(name, "@", result["watermark"], "evidence", result["evidence_seqs"], result["data"])
assert views["factual"]["data"][0]["subject"] == "repo.test_command"
assert any(row["status"] == "timeout" for row in views["progress"]["data"])

## Prompt assembly is another cached program

P0 is stable policy and tool schema. P1 is the history/checkpoint boundary. P2 contains task progress and relevant memory with view receipts. P3 contains the latest query and raw tail. Canonical field order and content hashes make identical inputs byte-stable. A new ledger event changes the watermark and dynamic view keys while leaving P0 unchanged.

In [ ]:
def assemble_prompt(query):
    progress = run_program("task.progress")
    memory = run_program("memory.semantic", query=query)
    episode = run_program("memory.episodic", limit=3)
    components = [
        {"tier": "P0", "content": {"worker": "coding-worker@7", "tools": ["python"]}},
        {"tier": "P1", "content": {"watermark": progress["watermark"]}},
        {
            "tier": "P2",
            "content": {"progress": progress["data"], "memory": memory["data"]},
            "views": [progress["cache_key"], memory["cache_key"]],
        },
        {
            "tier": "P3",
            "content": {"query": query, "recent": episode["data"]},
            "views": [episode["cache_key"]],
        },
    ]
    rendered = "\n\n".join(canonical(component) for component in components)
    return {
        "components": components,
        "bytes": rendered.encode(),
        "hash": hashlib.sha256(rendered.encode()).hexdigest(),
    }


prompt_a = assemble_prompt("recursive")
prompt_b = assemble_prompt("recursive")
assert prompt_a["bytes"] == prompt_b["bytes"]
p0_hash = hashlib.sha256(canonical(prompt_a["components"][0]).encode()).hexdigest()

db.execute(
    "INSERT INTO events VALUES (?, ?, ?, ?, ?, ?, ?)",
    (
        6,
        "2026-09-01T20:36:00Z",
        "file.write",
        "completed",
        "parser.py",
        "added recursion guard",
        canonical({"effect": "applied"}),
    ),
)
prompt_c = assemble_prompt("recursive")
assert prompt_c["hash"] != prompt_a["hash"]
assert hashlib.sha256(canonical(prompt_c["components"][0]).encode()).hexdigest() == p0_hash
print(
    {
        "initial_prompt_hash": prompt_a["hash"],
        "new_watermark_prompt_hash": prompt_c["hash"],
        "P0_unchanged": True,
    }
)

## Production mapping

Replace this in-memory SQLite fixture with the canonical JSONL or DuckDB ledger. Register restricted SQL programs through candidate → shadow → active → retired lifecycle gates. Cache by program version, ledger watermark, arguments, and retrieval implementation version. Keep source event IDs in every result. Use Lance only to locate evidence; hydrate final context from the canonical ledger.